In [1]:
import pandas as pd
df = pd.read_csv('D:/sathya/project/malicious_phish.csv')
print(df.head())
unique_types = df['type'].unique()
classified_df = pd.DataFrame(unique_types, columns=['Unique Types'])
print(classified_df)

                                                 url        type
0                                   br-icloud.com.br    phishing
1                mp3raid.com/music/krizz_kaliko.html      benign
2                    bopsecrets.org/rexroth/cr/1.htm      benign
3  http://www.garage-pirenne.be/index.php?option=...  defacement
4  http://adventure-nicaragua.net/index.php?optio...  defacement
  Unique Types
0     phishing
1       benign
2   defacement
3      malware


#preprocessing

In [2]:
import pandas as pd
df = pd.read_csv('D:/sathya/project/malicious_phish.csv')
print(df.head())

                                                 url        type
0                                   br-icloud.com.br    phishing
1                mp3raid.com/music/krizz_kaliko.html      benign
2                    bopsecrets.org/rexroth/cr/1.htm      benign
3  http://www.garage-pirenne.be/index.php?option=...  defacement
4  http://adventure-nicaragua.net/index.php?optio...  defacement


In [3]:
import re

def tokenize_url(url):
    tokens = re.split(r'\W+', url)
    tokens = [token for token in tokens if token]
    return tokens
df['tokens'] = df['url'].apply(tokenize_url)

In [4]:
from collections import Counter
all_tokens = [token for tokens in df['tokens'] for token in tokens]
vocabulary = Counter(all_tokens)
max_vocab_size = 5000
vocab = {token: idx for idx, (token, _) in enumerate(vocabulary.most_common(max_vocab_size), 1)}


In [5]:
def tokens_to_indices(tokens, vcab):
    return [vcab.get(token, 0) for token in tokens] 

df['token_indices'] = df['tokens'].apply(lambda tokens: tokens_to_indices(tokens, vocab))


In [6]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_seq_length = 10
padded_sequences = pad_sequences(df['token_indices'], maxlen=max_seq_length, padding='post', truncating='post')
padded_df = pd.DataFrame(padded_sequences)
print(padded_df.head())


      0     1     2     3    4   5  6    7   8   9
0    53  3826     1    53    0   0  0    0   0   0
1  1857     1   121     0    4   0  0    0   0   0
2     0     7     0  2794   15  14  0    0   0   0
3     2     3  3175     0  228   6  5    9  18  11
4     2  2309  3465    10    6   5  9  251  41  23


In [7]:
label_mapping = {'phishing': 0, 'benign': 1, 'defacement': 2, 'malware': 3}
df['label'] = df['type'].map(label_mapping)
X = padded_sequences
y = df['label'].values


In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


#Model

In [9]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense

model = Sequential([
    Embedding(input_dim=max_vocab_size + 1, output_dim=128),
    Conv1D(filters=128, kernel_size=1, activation='relu'),
    GlobalMaxPooling1D(),
    Dense(64, activation='relu'),
    Dense(4, activation='softmax')  # 4 classes: phishing, benign, defacement, malware
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [10]:
model.fit(X_train, y_train, epochs=1, batch_size=32, validation_split=0.1)

14652/14652 ━━━━━━━━━━━━━━━━━━━━ 235s 16ms/step - accuracy: 0.9243 - loss: 0.2131 - val_accuracy: 0.9500 - val_loss: 0.1407


In [11]:
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print(f'Test Accuracy: {test_accuracy:.4f}')

4070/4070 ━━━━━━━━━━━━━━━━━━━━ 31s 7ms/step - accuracy: 0.9487 - loss: 0.1429
Test Accuracy: 0.9487


In [12]:
y_pred = model.predict(X_test)
y_pred_classes = y_pred.argmax(axis=-1)


4070/4070 ━━━━━━━━━━━━━━━━━━━━ 23s 6ms/step


In [13]:
from sklearn.metrics import confusion_matrix, classification_report
conf_matrix = confusion_matrix(y_test, y_pred_classes)
print('Confusion Matrix:')
print(conf_matrix)
print('Classification Report:')
print(classification_report(y_test, y_pred_classes, target_names=label_mapping.keys()))

Confusion Matrix:
[[14491  4003   282    60]
 [ 1473 84249    41    15]
 [  227    38 18830     9]
 [  304   157    69  5991]]
Classification Report:
              precision    recall  f1-score   support

    phishing       0.88      0.77      0.82     18836
      benign       0.95      0.98      0.97     85778
  defacement       0.98      0.99      0.98     19104
     malware       0.99      0.92      0.95      6521

    accuracy                           0.95    130239
   macro avg       0.95      0.91      0.93    130239
weighted avg       0.95      0.95      0.95    130239



In [14]:
import numpy as np
conf_matrix = confusion_matrix(y_test, y_pred_classes)
print('Confusion Matrix:')
print(conf_matrix)

print('Classification Report:')
print(classification_report(y_test, y_pred_classes, target_names=label_mapping.keys()))

# Calculate accuracy from confusion matrix
correct_predictions = np.trace(conf_matrix)
total_predictions = np.sum(conf_matrix)
accuracy = correct_predictions / total_predictions
print(f'Calculated Accuracy from Confusion Matrix: {accuracy:.4f}')

Confusion Matrix:
[[14491  4003   282    60]
 [ 1473 84249    41    15]
 [  227    38 18830     9]
 [  304   157    69  5991]]
Classification Report:
              precision    recall  f1-score   support

    phishing       0.88      0.77      0.82     18836
      benign       0.95      0.98      0.97     85778
  defacement       0.98      0.99      0.98     19104
     malware       0.99      0.92      0.95      6521

    accuracy                           0.95    130239
   macro avg       0.95      0.91      0.93    130239
weighted avg       0.95      0.95      0.95    130239

Calculated Accuracy from Confusion Matrix: 0.9487


In [15]:
model.save('url_classifier_model.h5')

In [16]:
import pickle
with open("vocab.pkl", "wb") as f:
    pickle.dump(vocab, f)